# [Memory](https://langchain-ai.github.io/langgraph/concepts/memory/)
- LangGraph의 Memory는 AI가 대화 내용을 기억할 수 있게 해주는 기능입니다. 
- 사람이 대화할 때 이전에 말한 내용을 기억하는 것처럼, AI도 과거 대화를 기억해서 더 자연스러운 대화를 할 수 있게 해줍니다.


## 왜 Memory가 필요한가요?
> Memory가 없다면:
```text
    사용자: "내 이름은 김철수야"
    AI: "안녕하세요!"

    사용자: "내 이름이 뭐였지?"
    AI: "죄송해요, 모르겠습니다"
```
> Memory가 있다면:
```text
    사용자: "내 이름은 김철수야"
    AI: "안녕하세요 김철수님!"

    사용자: "내 이름이 뭐였지?"
    AI: "김철수님이라고 하셨어요!"
```


# ConversationBufferWindowMemory 예시
**최근 N개 메시지만 기억하는 효율적인 챗봇**

긴 대화에서는 모든 내용을 기억하면 메모리가 부족할 수 있어요. 그래서 최근 몇 개의 메시지만 기억하는 방법을 사용합니다!


### 1. AI 모델 설정

In [1]:
from langchain_ollama.chat_models import ChatOllama 

llm = ChatOllama(
    model="gemma3:4b",      # 이미 다운로드된 모델명 
    temperature=0.9,
    top_p=0.9,
    num_predict=512,
    keep_alive="10m"        # 로컬PC에서 모델이 메모리에 유지되는 시간 
)

### 2. 윈도우 메모리 노드 정의

In [2]:
from langgraph.graph import MessagesState

def chat_node_with_window(state: MessagesState, window_size=4):
    """최근 window_size 개의 메시지만 유지하는 함수"""
    messages = state["messages"]
    
    # 메시지가 너무 많으면 최근 것만 선택
    if len(messages) > window_size:
        # 시스템 메시지는 항상 유지하고, 나머지는 최근 것만
        system_messages = [msg for msg in messages if msg.type == "system"]
        other_messages = [msg for msg in messages if msg.type != "system"]
        
        # 최근 메시지만 선택
        recent_messages = other_messages[-window_size:]
        windowed_messages = system_messages + recent_messages
    else:
        windowed_messages = messages
    
    response = llm.invoke(windowed_messages)
    return {"messages": [response]}

### 3. 그래프 생성

In [3]:
from langgraph.graph import StateGraph, MessagesState, START, END

graph_window = StateGraph(MessagesState)
graph_window.add_node("chat", chat_node_with_window)
graph_window.add_edge(START, "chat")
graph_window.add_edge("chat", END)

### 4. 컴파일

In [4]:
from langgraph.checkpoint.memory import MemorySaver 

memory_window = MemorySaver()
app_with_window = graph_window.compile(checkpointer=memory_window)

print("Window Memory 챗봇이 준비되었습니다! (최근 4개 메시지만 기억)")

Window Memory 챗봇이 준비되었습니다! (최근 4개 메시지만 기억)


### 5. 테스트 

In [5]:
from langchain_core.messages import HumanMessage
import uuid

# 중요: 같은 thread_id를 사용해야 대화가 연결됩니다!
memory_id = str(uuid.uuid4())
# Window Memory 테스트: 오래된 정보를 기억할까요?
config_window = {"configurable": {"thread_id": memory_id}}

print("=== Window Memory 챗봇 테스트 ===")
print("많은 정보를 말해보고, 나중에 오래된 정보를 기억하는지 확인해보겠습니다!\n")

=== Window Memory 챗봇 테스트 ===
많은 정보를 말해보고, 나중에 오래된 정보를 기억하는지 확인해보겠습니다!



In [6]:
# 여러 정보 입력
messages_to_test = [
    "내 이름은 김철수야",
    "나는 30살이야", 
    "내 이름이 뭐였지?", # 기억함 
    "서울에 살고 있어",
    "취미는 축구야",
    "좋아하는 색깔은 파란색이야",
    "어제 영화를 봤어",
    "오늘은 운동을 했어",
    "내 이름이 뭐였지?",  # 오래전 정보 (기억 못할 수도 있음)
    "어제 뭐했다고 했지?"  # 최근 정보 (기억해야 함)
]

for i, msg in enumerate(messages_to_test, 1):
    print(f"{i}번째 대화: '{msg}'")
    result = app_with_window.invoke(
        {"messages": [HumanMessage(content=msg)]}, 
        config=config_window
    )
    print(f"AI: {result['messages'][-1].content}")
    print("-" * 40)

print("\n결과: 최근 4개 메시지만 기억해서 메모리가 효율적!")
print("오래된 정보(이름)는 잊어버릴 수 있지만, 최근 정보(어제 한 일)는 기억해요!")

1번째 대화: '내 이름은 김철수야'


/Users/gyoungwon-cho/dev/github/course_LLM/3. LangChain/1. colab/8. Memory/.venv/lib/python3.13/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=105c3626-b720-4c5d-9a22-76c994afe38f; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=ce85efc2-94c4-4a37-9bf2-190e8291eebb; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=face3246-6e0b-4fe4-bce0-d9a662949b4c; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=ea6e8dee-9254-4158-8693-53cbb96328a7; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=c03af59b-3d94-4603-ae75-16168bfde243; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=5a3f82cd-7983-49a2-b89

AI: 안녕하세요, 김철수님! 만나서 반갑습니다. 😊 

혹시 제가 무엇을 도와드릴까요?

----------------------------------------
2번째 대화: '나는 30살이야'


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=5a3f82cd-7983-49a2-b897-5656b5feaa19; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=de6acee6-1e80-4c95-a2ec-1ab9f7dbd09d; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=de6acee6-1e80-4c95-a2ec-1ab9f7dbd09d; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=c03af59b-3d94-4603-ae75-16168bfde243; trace=105c3626-b720-4c5d-9a22-76c994afe38f,id=105c3626-b720-4c5d-9a22-76c994afe38f; trace=c52203bd-1e69-42d6-99db-63ae2c120e05,id=c52203bd-1e69-42d6-99db-63ae2c120e05; trace=c52203bd-1e69-42d6-99db-63ae2c120e05,id=ca17bb96-52a6-4926-861d-c6d4704d9611; trace=c52203bd-1e69-42d6-99db-63ae2c120e05,id=3af2fac1-11ca-4f69-86a4-b4ee42ccb8b2; trace=c52203bd-1e69-42d6-99db-63ae2c120e05,id

AI: 오, 김철수님 30세시군요! 정말 멋진 나이시네요. 혹시 30대 생활에 대해 궁금한 점이 있으신가요, 아니면 요즘 관심 있는 분야가 있으신가요? 😊

----------------------------------------
3번째 대화: '내 이름이 뭐였지?'


...

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=f125c2e5-0a73-400e-831a-cf768918b942,id=1ddb21d0-0931-4b5c-853e-89154d6ce440; trace=f125c2e5-0a73-400e-831a-cf768918b942,id=680484c5-1716-4c1e-8ef9-c2b12d454694; trace=f125c2e5-0a73-400e-831a-cf768918b942,id=680484c5-1716-4c1e-8ef9-c2b12d454694; trace=f125c2e5-0a73-400e-831a-cf768918b942,id=0d0434a3-81f7-439a-b11c-acb374a6068d; trace=f125c2e5-0a73-400e-831a-cf768918b942,id=f125c2e5-0a73-400e-831a-cf768918b942; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=4aac56f8-189d-4776-82da-87d78b308f3c; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=658ab47b-55fb-42da-903e-f51c692d9f09; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=699efd07-9ea2-4ace-9be1-01d032b496af; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id

AI: 죄송합니다. 제가 이전 대화 내용을 기억하지 못합니다. 혹시 다시 알려주시겠어요? 😊
----------------------------------------
10번째 대화: '어제 뭐했다고 했지?'


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=c7b9ca6f-d45c-4019-88b8-21a2ce750b09; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=5234a2af-c25b-4d58-848e-0df52924de7f; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=5234a2af-c25b-4d58-848e-0df52924de7f; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=5b9039b0-c201-4b34-9ef4-66923bdafa4e; trace=4aac56f8-189d-4776-82da-87d78b308f3c,id=4aac56f8-189d-4776-82da-87d78b308f3c; trace=38012929-4e91-4777-9314-35fe58b20158,id=38012929-4e91-4777-9314-35fe58b20158; trace=38012929-4e91-4777-9314-35fe58b20158,id=b49919c9-2fca-41ec-a069-e8d3bff71554; trace=38012929-4e91-4777-9314-35fe58b20158,id=3f2f999c-f693-4371-960d-2d76f3fe5525; trace=38012929-4e91-4777-9314-35fe58b20158,id

AI: 음... 어제는 친구들과 영화를 보고, 저녁에는 맛있는 파스타를 먹고, 집에서 책을 읽었어요. 😊 기억이 잘 안 나네요. 😅

----------------------------------------

결과: 최근 4개 메시지만 기억해서 메모리가 효율적!
오래된 정보(이름)는 잊어버릴 수 있지만, 최근 정보(어제 한 일)는 기억해요!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=38012929-4e91-4777-9314-35fe58b20158,id=3370dc5f-2bd2-4c6e-82a4-5b5cb078190d; trace=38012929-4e91-4777-9314-35fe58b20158,id=e4dcdbf0-7b84-45e3-b370-e25e5c494e92; trace=38012929-4e91-4777-9314-35fe58b20158,id=e4dcdbf0-7b84-45e3-b370-e25e5c494e92; trace=38012929-4e91-4777-9314-35fe58b20158,id=ba576b0c-4cca-4014-a3de-2b0c7bd8ee8d; trace=38012929-4e91-4777-9314-35fe58b20158,id=38012929-4e91-4777-9314-35fe58b20158
